# Scenario Construction

Flows are grouped by start time, empty windows are omitted, and the final training target is interpreted as binary by the experiment pipeline.


In [ ]:
from pathlib import Path

import pandas as pd

from src.Scenario import (
    attack_block_info,
    build_attack_block,
    build_benign_baseline,
    build_scenario,
    create_windows,
    load_sample,
    prepare_scenario_attack_segments,
    rules_table,
)


In [ ]:
DATA_DIR = Path(r"data/CICDDoS2019/01-12/CSV")
OUTPUT_DIR = Path(r"output/Scenarios/CICDDoS2019")
WINDOW = "1s"
ATTACK_RATIO_THRESHOLD = 0.50
DNS_SAMPLE_ROWS = 1000
DNS_SAMPLE_START_ROW = 0
ATTACK_INSERT_DURATION = "5min"

BENIGN_SOURCE_FILES = [
    "DrDoS_DNS.csv", "DrDoS_LDAP.csv", "DrDoS_MSSQL.csv", "DrDoS_NetBIOS.csv",
    "DrDoS_NTP.csv", "DrDoS_SNMP.csv", "DrDoS_SSDP.csv", "DrDoS_UDP.csv",
    "Syn.csv", "TFTP.csv", "UDPLag.csv",
]

ATTACK_FILES = {
    "DNS": "DrDoS_DNS.csv",
    "LDAP": "DrDoS_LDAP.csv",
    "SYN": "Syn.csv",
}

SCENARIOS = {
    "Consistency": ["DNS", "DNS", "DNS"],
    "Generalization": ["DNS", "LDAP", "DNS"],
    "Adaptation": ["DNS", "SYN", "DNS"],
    "Recurrence": ["DNS", "SYN", "LDAP", "DNS", "SYN", "LDAP"],
}


## Windowing verification


In [ ]:
dnsSample = load_sample(
    DATA_DIR / ATTACK_FILES["DNS"],
    DNS_SAMPLE_START_ROW,
    DNS_SAMPLE_ROWS,
    WINDOW,
)

display(rules_table(dnsSample).query("ExistsInCSV").reset_index(drop=True))
display(create_windows(dnsSample, ATTACK_RATIO_THRESHOLD).head(30))


## Baseline and attack blocks


In [ ]:
baseline = build_benign_baseline(
    data_dir=DATA_DIR,
    source_files=BENIGN_SOURCE_FILES,
    output_path=OUTPUT_DIR / "baseline" / "benign_baseline_1s.csv",
    window=WINDOW,
    attack_ratio_threshold=ATTACK_RATIO_THRESHOLD,
)

attackBlocks = {}
attackBlockSummary = []

for attack, fileName in ATTACK_FILES.items():
    print("Processando:", attack)
    attackBlocks[attack] = build_attack_block(
        csv_path=DATA_DIR / fileName,
        attack_name=attack,
        output_path=OUTPUT_DIR / "attack_blocks" / f"{attack}_1s.csv",
        window=WINDOW,
        attack_ratio_threshold=ATTACK_RATIO_THRESHOLD,
    )
    attackBlockSummary.append(attack_block_info(attackBlocks[attack], attack))

display(pd.DataFrame(attackBlockSummary))


## Scenario generation


In [ ]:
attackSegments, segmentStrategy = prepare_scenario_attack_segments(
    attack_blocks=attackBlocks,
    scenarios=SCENARIOS,
    preferred_duration=ATTACK_INSERT_DURATION,
)

display(segmentStrategy)

scenarioResults = {}
for scenarioName, sequence in SCENARIOS.items():
    scenarioResults[scenarioName] = build_scenario(
        scenario_name=scenarioName,
        sequence=sequence,
        baseline=baseline,
        attack_segments=attackSegments,
        output_path=OUTPUT_DIR / "scenarios" / f"{scenarioName}_1s.csv",
        window=WINDOW,
    )
    print(scenarioName, "| janelas:", len(scenarioResults[scenarioName]))
